# Synthesis

## How to Use This Notebook

**1. Follow the numbered steps in order.**  
Each section builds upon the previous one, from setup, data loading, and climatology computation, to event analysis and visualization.

**2. Look for <font color="orange"> Orange cells  </font> and code cells marked as <font color="lightgreen">##### (User selection) ##### </font>:** 
| <font color="orange"> Orange cells  </font> | <font color="orange"> Need user intervention </font>|
| ----------- | ----------- |
| <font color="green">**Green cells** </font> | <font color="green">**Run automatically on user input provided in the orange cells and should not be adjusted in most cases** </font>|


**3. Run cells sequentially.**  
Start from the top and execute each cell (`Shift` + `Enter`).  

# <font color="#ffb612">PLEASE SWITCH TO AN APPROPRIATE R KERNEL FOR THIS NOTEBOOK</font>

In [ ]:
remotes::install_github("WorldWeatherAttribution/rwwa")
library(rwwa)

### <font color="orange"> User Input </font>

In [ ]:
your_save_directory <- "./data"

# Synthesise results from observations and climate models

### <font color="orange"> Import Observations </font>

In [ ]:
observations_file_name <- "res-obs_era5.csv"

models_file_name <- "res-models-Future-2.6.csv"

### <font color="green"> Autorun Cells </font>

In [ ]:
observations_path <- file.path(your_save_directory, observations_file_name)

# load the observational results
df_obs <- read.csv(observations_path, row.names = "X")

In [ ]:
df_obs

## Import Models

In [ ]:
models_path = file.path(your_save_directory, models_file_name)

# load the climate model results
df_models <- read.csv(models_path, row.names = "X")

In [ ]:
options(width = 10000)
options(max.print = 10000)
df_models

### 6.4 Filter models

In [ ]:
# Should we make a little R library as well? Is this easy to include in the py library?

filter_models <- function(df, include = c("true", "false", "both")) {
    include <- match.arg(include)

    if (include == "true") {
        df <- df[df$Include == TRUE, ]
    } else if (include == "false") {
    df <- df[df$Include == FALSE, ]
    } 
    # "both" returns df unchanged

    df
}


### <font color="orange"> Filter out excluded models </font>

In [ ]:
# options: "true", "false", "both".
# "true" includes only models where Include == TRUE, "false" includes only models where Include == FALSE, "both" includes all models.
models_to_include <- "true"

### <font color="green"> Autorun Cells </font>

In [ ]:
# Maybe change the name for this column

df_models$Include <- as.logical(df_models$Include)
# Only models with Include == TRUE
df_models <- filter_models(df_models, models_to_include)

In [ ]:
df_models

## Calculate Synthesis

### <font color="orange"> Choose Synthesis Type </font>

In [ ]:
# if looking at temperature data / 'shift' fit, use "abs";
# if looking at precipitation / 'fixeddisp', use "rel";
# use PR for probability ratio
synthesis_type <- "abs"

### <font color="green"> Autorun Cells </font>

In [ ]:
# change in intensity from past-present
synth_dI_attr <- synthesis(obs_in = df_obs[,grepl(paste0("dI.", synthesis_type), colnames(df_obs))], 
                           models_in = df_models[,grepl(paste0("attr_dI.", synthesis_type), colnames(df_models))], 
                           synth_type = synthesis_type)
synth_dI_attr
# if you see error/warning messages below, you probably have infinite best estimates in your observations 

In [ ]:

# change in likelihood from past-present (if there are any infinite values in the PRs, replace them with estimated values)
synth_PR_attr <- synthesis(obs_in = infer_infinite(df_obs[,grepl("PR", colnames(df_obs))]), 
                           models_in = infer_infinite(df_models[,grepl("attr_PR", colnames(df_models))]), 
                           synth_type = "PR")
synth_PR_attr
# if you see error/warning messages below, you probably have infinite best estimates in your observations 

In [ ]:
# change in intensity from present-future
synth_dI_proj <- synthesis(obs_in = NA, 
                           models_in = df_models[,grepl(paste0("proj_dI.", synthesis_type), colnames(df_models))], 
                           synth_type = synthesis_type)
synth_dI_proj

In [ ]:
# change in likelihood from present-future
synth_PR_proj <- synthesis(obs_in = NA, 
                           models_in = df_models[,grepl("proj_PR", colnames(df_models))], 
                           synth_type = "PR")
synth_PR_proj

### Combine synthesis results into table

In [ ]:
# function for extracting the correct rows from the synthesis

extract_row <- function(df, group, best = TRUE, use_wb = FALSE) {
  row <- df[df$group == group, ]

  data.frame(
    best = if (best) row$est else NA_real_,
    low  = if (use_wb) row$l_wb else row$lower,
    high = if (use_wb) row$u_wb else row$upper
  )
}


In [ ]:
# extract rows and combine into new table

obs_PR <- extract_row(synth_PR_attr$df, "obs_synth")
obs_dI <- extract_row(synth_dI_attr$df, "obs_synth")

model_PR <- extract_row(synth_PR_attr$df, "model_synth")
model_dI <- extract_row(synth_dI_attr$df, "model_synth")

syn_w_PR <- extract_row(synth_PR_attr$df, "synth")
syn_w_dI <- extract_row(synth_dI_attr$df, "synth")

syn_uw_PR <- extract_row(synth_PR_attr$df, "synth", best = FALSE, use_wb = TRUE)
syn_uw_dI <- extract_row(synth_dI_attr$df, "synth", best = FALSE, use_wb = TRUE)

model_f_PR <- extract_row(synth_PR_proj$df, "model_synth")
model_f_dI <- extract_row(synth_dI_proj$df, "model_synth")

# combine results into new table
synth_comb <- rbind(
  obs = cbind(PR = obs_PR, dI = obs_dI),
  model = cbind(PR = model_PR, dI = model_dI),
  `synthesis weighted` = cbind(PR = syn_w_PR, dI = syn_w_dI),
  `synthesis unweighted` = cbind(PR = syn_uw_PR, dI = syn_uw_dI),
  `model future` = cbind(PR = model_f_PR, dI = model_f_dI)
)

colnames(synth_comb) <- c(
  "PR best", "PR low", "PR high",
  "dI best", "dI low", "dI high"
)

# set the 'best' values for the unweighted synthesis
synth_comb["synthesis unweighted", "PR best"] <- synth_PR_attr$uw_mean
synth_comb["synthesis unweighted", "dI best"] <- synth_dI_attr$uw_mean

synth_comb

### <font color="orange"> Optionally Use Different Filenames </font>

In [ ]:
# save all the synthesised results
write.csv(synth_dI_attr$df, file.path(your_save_directory, "synth_dI_attr.csv"))
write.csv(synth_PR_attr$df, file.path(your_save_directory, "synth_PR_attr.csv"))
write.csv(synth_dI_proj$df, file.path(your_save_directory, "synth_dI_proj.csv"))
write.csv(synth_PR_proj$df, file.path(your_save_directory, "synth_PR_proj.csv"))
write.csv(synth_comb, file.path(your_save_directory, "synth_comb.csv"))

## Optionally calculate the inverse PR

### Observations

In [ ]:
# Select columns containing "PR_"
df_obs_inverse_pr  <- df_obs[, grepl("PR_", names(df_obs)), drop = FALSE]

# Invert values (1 / value)
df_obs_inverse_pr[] <- 1 / df_obs_inverse_pr

df_obs_inverse_pr

### Models

In [ ]:
# Select columns containing "PR_"
df_models_inverse_pr  <- df_models[, grepl("PR_", names(df_models)), drop = FALSE]

# Invert values (1 / value)
df_models_inverse_pr[] <- 1 / df_models_inverse_pr

df_models_inverse_pr

### Synthesis Attributes

In [ ]:
synth_PR_attr_inverse_pr <- synth_PR_attr$df

num_cols <- sapply(synth_PR_attr_inverse_pr, is.numeric)

synth_PR_attr_inverse_pr[, num_cols] <- 1 / synth_PR_attr_inverse_pr[, num_cols]

synth_PR_attr_inverse_pr


### Synthesis Projection

In [ ]:
synth_PR_proj_inverse_pr <- synth_PR_proj$df

num_cols <- sapply(synth_PR_proj_inverse_pr, is.numeric)

synth_PR_proj_inverse_pr[, num_cols] <- 1 / synth_PR_proj_inverse_pr[, num_cols]

synth_PR_proj_inverse_pr

## Synthesis figures

### Optionally remove models from probability if all values are INF



In [ ]:
# Models that have al inf values for PR should be excluded from the probability ratio but should be included in the intensity
remove_inf <- TRUE

if (remove_inf) {

  synth_PR_attr$df <- synth_PR_attr$df[
    !is.infinite(synth_PR_attr$df$est) &
    !is.infinite(synth_PR_attr$df$lower),
  ]

  synth_PR_proj$df <- synth_PR_proj$df[
    !is.infinite(synth_PR_proj$df$est) &
    !is.infinite(synth_PR_proj$df$lower),
  ]
}

In [ ]:
prep_rc = c(1, 2)
prep_h = 5                      # height of the figure (ins)
prep_w = 5                      # width of the figure (ins)
prep_res = 200
prep_pch = 20
prep_oma = c(0, 14, 0, 0)       # increase second number until model names fit in margin
prep_mar = c(3, .5, 2, .5)      # shouldn't need to be changed

# Change until the bars fit the plot
x_lim_dI <- range(
  synth_dI_attr$df[sapply(synth_dI_attr$df, is.numeric)],
  na.rm = TRUE
) * 1.1
x_lim_PR = range(
  synth_PR_attr$df[sapply(synth_PR_attr$df, is.numeric)],
  na.rm = TRUE
) * 1.1 # add some padding to the x-axis limits

In [ ]:
# put two figures next to each other
prep_window(prep_rc,
            h = prep_h,         # height of the figure (ins)
            w = prep_w,         # width of each panel (ins)
            res = prep_res,
            pch = prep_pch,
            oma = prep_oma,     # width of each panel (ins)
            mar = prep_mar)     # shouldn't need to be changed

# set the x-axis (xlim) so that both the past & future changes use the same scaling
plot_synthesis(synth_dI_attr, lwd=12, add_space = F, main = "(a) Change in intensity", xlim = x_lim_dI)
plot_synthesis(synth_PR_attr, lwd=12, add_space = F, hide_labels = T, main = "(b) Probability ratio", xlim = x_lim_PR)

In [ ]:
attr_export_path <- file.path(your_save_directory, "synth-fig_attr.png")

png(attr_export_path, height = 360, width = 480*1.5); par(mfrow = prep_rc, oma = prep_oma, mar = prep_mar); {
    # set the x-axis (xlim) so that both the past & future changes use the same scaling
    plot_synthesis(synth_dI_attr, add_space = F, main = "(a) Change in intensity", xlim = x_lim_dI)
    plot_synthesis(synth_PR_attr, add_space = F, hide_labels = T, main = "(b) Probability ratio", xlim = x_lim_PR)
}; dev.off()

In [ ]:
prep_rc = c(1, 2)
prep_h = 5                      # height of the figure (ins)
prep_w = 5                      # width of the figure (ins)
prep_res = 200
prep_pch = 20
prep_oma = c(0, 14, 0, 0)       # increase second number until model names fit in margin
prep_mar = c(3, .5, 2, .5)      # shouldn't need to be changed

# Change until the bars fit the plot
x_lim_dI <- range(
  synth_dI_proj$df[sapply(synth_dI_proj$df, is.numeric)],
  na.rm = TRUE
) * 1.1
x_lim_PR = range(
  synth_PR_proj$df[sapply(synth_PR_proj$df, is.numeric)],
  na.rm = TRUE
) * 1.1 # add some padding to the x-axis limits

In [ ]:
prep_window(prep_rc,
            h = prep_h,         # height of the figure (ins)
            w = prep_w,         # width of each panel (ins)
            res = prep_res,
            pch = prep_pch,
            oma = prep_oma,     # increase second number until model names fit in margin
            mar = prep_mar)     # shouldn't need to be changed

# set the x-axis (xlim) so that both the past & future changes use the same scaling
plot_synthesis(synth_dI_proj, lwd=12, add_space = F, main = "(a) Change in intensity", xlim = x_lim_dI)
plot_synthesis(synth_PR_proj, lwd=12, add_space = F, hide_labels = T, main = "(b) Probability ratio", xlim = x_lim_PR)

In [ ]:
proj_export_path <- file.path(your_save_directory, "synth-fig_proj.png")

png(proj_export_path, height = 240, width = 480*1.5); par(mfrow = prep_rc, oma = prep_oma, mar = prep_mar); {
    # set the x-axis (xlim) so that both the past & future changes use the same scaling
    plot_synthesis(synth_dI_proj, add_space = F, main = "(a) Change in intensity", xlim = x_lim_dI)
    plot_synthesis(synth_PR_proj, add_space = F, hide_labels = T, main = "(b) Probability ratio", xlim = x_lim_PR)
}; dev.off()